<a href="https://colab.research.google.com/github/kayla-m20/Data-Analyst-Intern-at-Glints---Technical-Assignment/blob/main/notebooks/assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

os.makedirs("charts", exist_ok=True)
# ============ 0. DATA PREPARATION ============
import pandas as pd, numpy as np

X = "https://github.com/kayla-m20/Data-Analyst-Intern-at-Glints---Technical-Assignment/raw/cc61ed070021550d2f62f45c038e1f097939ac1a/Data%20Source%202.xlsx"
d = pd.read_excel(X, "staff_particulars")
rc = pd.read_excel(X, "recruitment_costs")
d["Hispanic/Latino"] = d["Hispanic/Latino"].str.title()
SNAP = pd.Timestamp("2016-06-17")  # reference date = day after last recorded termination (16 Jun 2016)
d["status"] = d["Employment Status"]
d["term"] = d["Date of Termination"]
d["end"] = d["term"].fillna(SNAP)
d["tenure"] = ((d["end"] - d["Date of Hire"]).dt.days / 365.25).clip(lower=0)
d["age"] = (d["end"] - d["DOB"]).dt.days / 365.25
d["age_now"] = (SNAP - d["DOB"]).dt.days / 365.25
d["hire_year"] = d["Date of Hire"].dt.year
d["term_year"] = d["term"].dt.year
d["vol"] = (d.status == "Voluntarily Terminated").astype(int)
d["cause"] = (d.status == "Terminated for Cause").astype(int)
d["left"] = d.status.isin(["Voluntarily Terminated", "Terminated for Cause"]).astype(int)
perf_map = {"Exceptional": 5, "Exceeds": 4, "Fully Meets": 3, "90-day meets": 3, "Needs Improvement": 2, "PIP": 1}
d["perf"] = d["Performance Score"].map(perf_map)
# analysis population: excl. 'Future Start' and anyone hired after snapshot
E = d[(d.status != "Future Start") & (d["Date of Hire"] <= SNAP)].copy()

# ============ 1-4. EDA (A), sources (B), managers (C), pay drivers (D) + charts c1-c6 ============
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf, pickle, json

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.titlesize": 10.5,
    }
)
NAVY = "#1F3A5F"
TEAL = "#2A9D8F"
ORG = "#E76F51"
GRY = "#8D99AE"
GOLD = "#E9C46A"
LGT = "#D9E2EC"
R = {}
E["rated"] = E["Performance Score"].isin(["Exceptional", "Exceeds", "Fully Meets", "Needs Improvement", "PIP"])
E["hi"] = E["Performance Score"].isin(["Exceptional", "Exceeds"]).astype(int)
E["lo"] = E["Performance Score"].isin(["Needs Improvement", "PIP"]).astype(int)
cur = E[E.status.isin(["Active", "Leave of Absence"])].copy()
R["n_E"] = len(E)
R["n_cur"] = len(cur)
# ---------- A. EDA ----------
yrs = range(2010, 2017)
hc = []
hires = []
exits = []
vex = []
for y in yrs:
    t = pd.Timestamp(f"{y}-12-31") if y < 2016 else SNAP
    hc.append(int(((E["Date of Hire"] <= t) & ((E.term.isna()) | (E.term > t))).sum()))
    hires.append(int((E.hire_year == y).sum()))
    exits.append(int((E.term_year == y).sum()))
    vex.append(int(((E.term_year == y) & (E.vol == 1)).sum()))
R["trend"] = dict(years=list(yrs), hc=hc, hires=hires, exits=exits, vex=vex)
fig, ax = plt.subplots(figsize=(7.2, 3.3))
x = np.arange(len(yrs))
w = 0.38
ax.bar(x - w / 2, hires, w, color=TEAL, label="New hires")
ax.bar(x + w / 2, exits, w, color=ORG, label="Exits")
for i, v in enumerate(hires):
    ax.text(x[i] - w / 2, v + 1.5, str(v), ha="center", fontsize=7.5, color=TEAL)
for i, v in enumerate(exits):
    ax.text(x[i] + w / 2, v + 1.5, str(v), ha="center", fontsize=7.5, color=ORG)
ax.set_xticks(x)
ax.set_xticklabels([str(y) if y < 2016 else "2016\n(to 17 Jun)" for y in yrs])
ax.set_ylim(0, 110)
ax2 = ax.twinx()
ax2.plot(x, hc, color=NAVY, marker="o", lw=2, label="Headcount at year-end")
ax2.set_ylim(0, 260)
ax2.spines["right"].set_visible(True)
for i, v in enumerate(hc):
    ax2.annotate(
        v,
        (x[i], v),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        color=NAVY,
        fontsize=8,
        fontweight="bold",
    )
ax.set_ylabel("People hired / exited")
ax2.set_ylabel("Headcount")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc="lower center", frameon=False, fontsize=8, ncol=3)
ax.set_title("Hiring, exits and headcount by year")
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.savefig("charts/c1_trend.png", dpi=200)
plt.close()
# dept turnover
dg = E.groupby("Department").agg(
    n=("vol", "size"),
    cur=("status", lambda s: s.isin(["Active", "Leave of Absence"]).sum()),
    vol=("vol", "sum"),
    cause=("cause", "sum"),
    pay=("Pay Rate", "mean"),
    fem=("Sex", lambda s: (s == "Female").mean()),
)
dg["vol_rate"] = dg.vol / dg.n
dg = dg.sort_values("vol_rate", ascending=False)
R["dept"] = dg
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={"width_ratios": [1, 1]})
a = axs[0]
dd = dg.drop("Executive Office")
a.barh(dd.index[::-1], dd.vol_rate[::-1] * 100, color=[ORG if v > 0.3 else GRY for v in dd.vol_rate[::-1]])
for i, (v, n) in enumerate(zip(dd.vol_rate[::-1], dd.n[::-1])):
    a.text(v * 100 + 1, i, f"{v*100:.0f}%  (n={n})", va="center", fontsize=8)
a.set_xlim(0, 60)
a.set_xlabel("% of all employees who resigned")
a.set_title("Resignation rate by department")
b = axs[1]
cd = cur.Department.value_counts()
b.barh(cd.index[::-1], cd.values[::-1], color=NAVY)
for i, v in enumerate(cd.values[::-1]):
    b.text(v + 2, i, str(v), va="center", fontsize=8)
b.set_xlim(0, 130)
b.set_xlabel("Current employees")
b.set_title("Current headcount (188)")
plt.tight_layout()
plt.savefig("charts/c2_dept.png", dpi=200)
plt.close()
# demographics panel
fig, axs = plt.subplots(2, 2, figsize=(7.2, 4.8))
a = axs[0, 0]
ab = (
    pd.cut(cur.age_now, [0, 30, 40, 50, 60, 100], labels=["<30", "30-39", "40-49", "50-59", "60+"])
    .value_counts()
    .sort_index()
)
a.bar(ab.index.astype(str), ab.values, color=NAVY)
[a.text(i, v + 1, str(v), ha="center", fontsize=8) for i, v in enumerate(ab.values)]
a.set_title("Current employees by age")
a.set_ylim(0, ab.max() * 1.15)
a = axs[0, 1]
rc_ = (
    cur.RaceDesc.replace(
        {
            "American Indian or Alaska Native": "Native American",
            "Black or African American": "Black / African Am.",
            "Two or more races": "Two or more",
        }
    ).value_counts(normalize=True)
    * 100
)
a.barh(rc_.index[::-1], rc_.values[::-1], color=TEAL)
[a.text(v + 1, i, f"{v:.0f}%", va="center", fontsize=8) for i, v in enumerate(rc_.values[::-1])]
a.set_title("Current employees by race")
a.set_xlim(0, 80)
a = axs[1, 0]
mg = E.groupby("MaritalDesc").agg(n=("vol", "size"), r=("vol", "mean")).sort_values("r", ascending=False)
a.bar(mg.index, mg.r * 100, color=[ORG if r > 0.3 else GRY for r in mg.r])
[
    a.text(i, r * 100 + 1, f"{r*100:.0f}%\n(n={n})", ha="center", fontsize=7.5)
    for i, (r, n) in enumerate(zip(mg.r, mg.n))
]
a.set_ylim(0, 70)
a.set_title("Resignation rate by marital status")
a.tick_params(axis="x", labelsize=8)
a = axs[1, 1]
rs = d[d.vol == 1]["Reason For Term"].value_counts().head(7)
lab = {
    "Another position": "Another position",
    "unhappy": "Unhappy",
    "more money": "More money",
    "career change": "Career change",
    "hours": "Hours",
    "return to school": "Return to school",
    "relocation out of area": "Relocation",
}
a.barh([lab.get(i, i) for i in rs.index[::-1]], rs.values[::-1], color=GOLD)
[a.text(v + 0.3, i, str(v), va="center", fontsize=8) for i, v in enumerate(rs.values[::-1])]
a.set_title("Why people resign (88 exits)")
a.set_xlim(0, 24)
plt.tight_layout()
plt.savefig("charts/c3_demo.png", dpi=200)
plt.close()
# misc numbers
R["sex_cur"] = cur.Sex.value_counts(normalize=True).to_dict()
R["age_cur"] = cur.age_now.describe().to_dict()
R["ten_cur_med"] = float(cur.tenure.median())
R["state_ma"] = float((cur.State == "MA").mean())
R["cit"] = cur.CitizenDesc.value_counts().to_dict()
R["leaver_ten_med"] = float(E[E.left == 1].tenure.median())
R["vol_ten_by_dept"] = E[E.vol == 1].groupby("Department").tenure.median().to_dict()
R["vol_total"] = int(E.vol.sum())
R["left_total"] = int(E.left.sum())
R["pay_sex"] = E.groupby("Sex")["Pay Rate"].mean().to_dict()
mgmt = E.Position.str.contains("Manager|Director|CIO|CEO|President")
R["mgmt_fem"] = float((E[mgmt].Sex == "Female").mean())
R["mgmt_n"] = int(mgmt.sum())
R["perf_cur"] = cur["Performance Score"].value_counts().to_dict()
R["age60"] = E[E.age_now >= 60].vol.mean()
R["reasons"] = d[d.vol == 1]["Reason For Term"].value_counts().to_dict()
R["tenure_exit_q"] = E[E.left == 1].tenure.quantile([0.25, 0.5, 0.75]).to_dict()
# ---------- B. Sources ----------
dr = E.groupby("Department").agg(ev=("vol", "sum"), yrs=("tenure", "sum"))
dr["rate"] = dr.ev / dr.yrs
E["exp"] = E.Department.map(dr.rate) * E.tenure
cost = rc.set_index("Employment Source")["Total"]
sr = E.groupby("Employee Source").agg(
    n=("vol", "size"),
    vol=("vol", "sum"),
    yrs=("tenure", "sum"),
    exp=("exp", "sum"),
    perf=("perf", "mean"),
    hi=("hi", "mean"),
)
sr["perf"] = E[E.rated].groupby("Employee Source").perf.mean()
sr["nr"] = E[E.rated].groupby("Employee Source").size()
sr["oe"] = sr.vol / sr.exp
sr["rate100"] = sr.vol / sr.yrs * 100
sr["cost"] = cost.reindex(sr.index)
sr["cph"] = sr.cost / sr.n
el = E[E["Date of Hire"] <= SNAP - pd.Timedelta(days=365)].copy()
el["stay12"] = ((el.term.isna()) | ((el.term - el["Date of Hire"]).dt.days >= 365)).astype(int)
sr["stay12"] = el.groupby("Employee Source").stay12.mean()
elig = sr[sr.n >= 10].copy()


def pr(s, asc=True):
    return s.rank(pct=True, ascending=asc) * 100  # higher = better


elig["s_ret"] = pr(elig.oe, False)
elig["s_perf"] = pr(elig.perf, True)
elig["s_cost"] = pr(elig.cph, False)
elig["score"] = 0.40 * elig.s_ret + 0.35 * elig.s_perf + 0.25 * elig.s_cost
elig = elig.sort_values("score", ascending=False)
R["src"] = elig
R["src_small"] = sr[sr.n < 10]
fig, ax = plt.subplots(figsize=(7.2, 4.3))
cols = [TEAL if s >= 65 else (ORG if s <= 35 else GRY) for s in elig.score[::-1]]
ax.barh(
    elig.index[::-1]
    .str.replace("Search Engine - Google Bing Yahoo", "Search Engine (G/B/Y)")
    .str.replace("Social Networks - Facebook Twitter etc", "Social Networks")
    .str.replace("Newspager/Magazine", "Newspaper/Magazine"),
    elig.score[::-1],
    color=cols,
)
for i, (s, n) in enumerate(zip(elig.score[::-1], elig.n[::-1])):
    ax.text(s + 1, i, f"{s:.0f}  (n={n})", va="center", fontsize=8)
ax.set_xlim(0, 100)
ax.set_xlabel("Composite score (0-100): retention 40% + performance 35% + cost 25%")
ax.set_title("Recruitment source scorecard (sources with 10+ hires)")
plt.tight_layout()
plt.savefig("charts/c4_sources.png", dpi=200)
plt.close()
# ---------- C. Managers ----------
M = E[E["Manager Name"] != "Board of Directors"]
mg = M.groupby("Manager Name").agg(
    n=("vol", "size"),
    cur=("status", lambda s: s.isin(["Active", "Leave of Absence"]).sum()),
    yrs=("tenure", "sum"),
    obs=("vol", "sum"),
    exp=("exp", "sum"),
    cause=("cause", "sum"),
)
mg["oe"] = mg.obs / mg.exp
mg["p_hi"] = [stats.poisson.sf(o - 1, e) for o, e in zip(mg.obs, mg.exp)]
mg["perf"] = M[M.rated].groupby("Manager Name").perf.mean()
mg["prob"] = M.groupby("Manager Name").apply(lambda g: ((g.lo == 1) | (g.cause == 1)).mean())
mg["dept"] = M.groupby("Manager Name").Department.agg(lambda x: x.value_counts().index[0])
el_ = mg[mg.n >= 10].copy()
z = lambda s: (s - s.mean()) / s.std()
el_["score"] = 0.5 * z(-el_.oe) + 0.3 * z(el_.perf) + 0.2 * z(-el_.prob)
el_["score100"] = 50 + el_.score * 15  # display scale
el_ = el_.sort_values("score", ascending=False)
R["mgr"] = el_
R["mgr_small"] = mg[mg.n < 10]
fig, ax = plt.subplots(figsize=(7.2, 4.0))
lab = [f"{m} ({r.dept.split()[0]})" for m, r in el_.iterrows()]
colors = [TEAL if i < 3 else (ORG if i >= len(el_) - 3 else GRY) for i in range(len(el_))]
ax.barh(lab[::-1], el_.score100[::-1], color=colors[::-1])
ax.axvline(50, color="k", lw=0.8, ls="--")
ax.text(50.5, len(el_) - 0.4, "average", fontsize=7.5)
for i, s in enumerate(el_.score100[::-1]):
    ax.text(
        s + 0.6 if s >= 50 else s - 0.6, i, f"{s:.0f}", va="center", ha="left" if s >= 50 else "right", fontsize=8
    )
ax.set_xlim(20, 85)
ax.set_xlabel("Manager score (50 = average of the 14 managers with 10+ reports)")
ax.set_title("Manager scorecard")
plt.tight_layout()
plt.savefig("charts/c5_mgr.png", dpi=200)
plt.close()
# ---------- D. Pay ----------
P = E.copy()
P = P.rename(
    columns={
        "Pay Rate": "pay",
        "Employee Source": "src",
        "Manager Name": "mgr",
        "RaceDesc": "race",
        "MaritalDesc": "marital",
        "Performance Score": "ps",
        "CitizenDesc": "cit",
    }
)
P["lp"] = np.log(P.pay)
vs = {
    "Job title (Position)": "C(Position)",
    "Department": "C(Department)",
    "Manager": "C(mgr)",
    "Recruitment source": "C(src)",
    "Race": "C(race)",
    "Performance score": "C(ps)",
    "Gender": "C(Sex)",
    "Marital status": "C(marital)",
    "Citizenship": "C(cit)",
    "Age": "age",
    "Tenure": "tenure",
}
alone = {k: smf.ols("pay ~ " + v, P).fit().rsquared for k, v in vs.items()}
base = smf.ols("pay ~ C(Position)", P).fit()
extra = {
    k: smf.ols("pay ~ C(Position)+" + v, P).fit().rsquared - base.rsquared
    for k, v in vs.items()
    if k not in ("Job title (Position)",)
}
R["pay_alone"] = alone
R["pay_extra"] = extra
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ks = sorted(alone, key=alone.get)
colors_alone = [NAVY if alone[k] > 0.5 else GRY for k in ks]
ax.barh(ks, [alone[k] * 100 for k in ks], color=colors_alone)
for i, k in enumerate(ks):
    ax.text(alone[k] * 100 + 1, i, f"{alone[k]*100:.0f}%", va="center", fontsize=8)
ax.set_xlim(0, 110)
ax.set_title("Pay variation explained by each factor alone")
ax.set_xlabel("% of differences in hourly pay")
plt.tight_layout()
plt.savefig("charts/c6a_pay_alone.png", dpi=200)
plt.close()

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ke = sorted(extra, key=extra.get)
colors_extra = [TEAL if extra[k] > 0.0005 else GRY for k in ke]
ax.barh(ke, [extra[k] * 100 for k in ke], color=colors_extra)
for i, k in enumerate(ke):
    ax.text(extra[k] * 100 + 0.02, i, f"+{extra[k]*100:.2f}%", va="center", fontsize=7.5)
ax.set_xlim(0, 0.75)
ax.set_title("Extra explained once job title is known")
ax.set_xlabel("% points added")
plt.tight_layout()
plt.savefig("charts/c6b_pay_extra.png", dpi=200)
plt.close()
raw = smf.ols("lp ~ C(Sex)", P).fit()
adj = smf.ols("lp ~ C(Sex)+C(Position)", P).fit()
adj2 = smf.ols("lp ~ C(Sex)+C(Position)+tenure+age", P).fit()
R["gap_raw"] = float(np.exp(raw.params["C(Sex)[T.Male]"]) - 1)
R["gap_adj"] = float(np.exp(adj.params["C(Sex)[T.Male]"]) - 1)
R["gap_adj_p"] = float(adj.pvalues["C(Sex)[T.Male]"])
R["gap_adj2"] = float(np.exp(adj2.params["C(Sex)[T.Male]"]) - 1)
tn = smf.ols("pay ~ C(Position)+tenure", P).fit()
R["tenure_coef"] = float(tn.params["tenure"])
R["tenure_p"] = float(tn.pvalues["tenure"])
# grid check
sg = pd.read_excel(X, "salary_grid", header=None, skiprows=2)
sg.columns = ["Position", "smin", "smid", "smax", "hmin", "hmid", "hmax"]
G = E.merge(sg, on="Position", how="inner")
G["gp"] = np.select(
    [G["Pay Rate"] < G.hmin, G["Pay Rate"] > G.hmax, G["Pay Rate"] <= G.hmid],
    ["Below grid", "Above grid", "Lower half"],
    "Upper half",
)
R["grid_n"] = len(G)
R["grid"] = G.gp.value_counts().to_dict()
R["grid_by_pos"] = G[G.gp.isin(["Below grid", "Above grid"])].groupby(["Position", "gp"]).size().to_dict()
R["no_grid_pos"] = sorted(set(E.Position) - set(sg.Position))
R["pay_pos"] = P.groupby("Position").pay.agg(["count", "mean", "std", "min", "max"])
pickle.dump(R, open("R.pkl", "wb"))
# prints
print(R["trend"])
print(R["sex_cur"], R["age_cur"]["mean"], R["ten_cur_med"], R["state_ma"])
print(R["dept"].round(3))
print(R["leaver_ten_med"], R["vol_ten_by_dept"], R["mgmt_fem"], R["mgmt_n"], R["pay_sex"])
print(R["age60"], R["tenure_exit_q"])
print(R["src"][["n", "oe", "rate100", "perf", "cph", "stay12", "s_ret", "s_perf", "s_cost", "score"]].round(2))
print(R["src_small"][["n", "vol", "perf", "cph"]])
print(R["mgr"][["dept", "n", "cur", "obs", "exp", "oe", "p_hi", "perf", "prob", "cause", "score100"]].round(2))
print(R["mgr_small"][["dept", "n", "obs", "exp", "perf", "prob"]].round(2))
print({k: round(v, 4) for k, v in alone.items()}, {k: round(v, 5) for k, v in extra.items()})
print(R["gap_raw"], R["gap_adj"], R["gap_adj_p"], R["gap_adj2"], R["tenure_coef"], R["tenure_p"])
print(R["grid_n"], R["grid"], R["grid_by_pos"], R["no_grid_pos"])

# ============ 5. ATTRITION (E): person-quarter panel ============
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

pm = E.groupby("Position")["Pay Rate"].transform("median")
E["pay_vs_pos"] = E["Pay Rate"] / pm  # pay relative to median of own job title
rows = []
for t0 in pd.date_range("2011-01-01", "2016-01-01", freq="QS"):
    t1 = t0 + pd.DateOffset(months=3)
    r = E[(E["Date of Hire"] <= t0) & ((E.term.isna()) | (E.term > t0))].copy()
    r["t0"] = t0
    r["y"] = ((r.term.notna()) & (r.term <= t1)).astype(int)
    r["ten0"] = (t0 - r["Date of Hire"]).dt.days / 365.25
    r["age0"] = (t0 - r.DOB).dt.days / 365.25
    rows.append(r)
P = pd.concat(rows).reset_index(drop=True)


def feats(df, t0col=None):
    f = pd.DataFrame(index=df.index)
    f["production"] = (df.Department == "Production").astype(int)
    f["tenure_lt6m"] = (df.ten0 < 0.5).astype(int)
    f["tenure_6m_1y"] = ((df.ten0 >= 0.5) & (df.ten0 < 1)).astype(int)
    f["tenure_3y_plus"] = (df.ten0 >= 3).astype(int)
    f["age_50_plus"] = (df.age0 >= 50).astype(int)
    f["divorced"] = (df.MaritalDesc == "Divorced").astype(int)
    f["married"] = (df.MaritalDesc == "Married").astype(int)
    f["pay_below_pos_median"] = (df.pay_vs_pos < 0.97).astype(int)
    f["perf_weak"] = df["Performance Score"].isin(["Needs Improvement", "PIP"]).astype(int)
    f["perf_90day"] = (df["Performance Score"] == "90-day meets").astype(int)
    f["perf_top"] = df["Performance Score"].isin(["Exceptional", "Exceeds"]).astype(int)
    f["male"] = (df.Sex == "Male").astype(int)
    f["non_citizen"] = (df.CitizenDesc != "US Citizen").astype(int)
    return f


F = feats(P)
y = P.y.values
g = P["Employee Number"].astype(str) + P["Employee Name"]

# ============ 5E-2. validation, scoring of current staff, segment chart ============
cols = [
    "production",
    "tenure_3y_plus",
    "age_50_plus",
    "divorced",
    "married",
    "pay_below_pos_median",
    "perf_weak",
    "perf_90day",
]
oof = np.zeros(len(P))
for tr, te in GroupKFold(5).split(F, y, g):
    m_ = LogisticRegression(C=0.3, max_iter=2000).fit(F.iloc[tr][cols], y[tr])
    oof[te] = m_.predict_proba(F.iloc[te][cols])[:, 1]
auc_cv = roc_auc_score(y, oof)
aucs = []
for yr in [2013, 2014, 2015, 2016]:
    tr = P.t0 < f"{yr}-01-01"
    te = (P.t0 >= f"{yr}-01-01") & (P.t0 < f"{yr+1}-01-01")
    if y[te].sum() > 0 and te.sum() > 0:
        m_ = LogisticRegression(C=0.3, max_iter=2000).fit(F[tr][cols], y[tr])
        aucs.append(roc_auc_score(y[te], m_.predict_proba(F[te][cols])[:, 1]))
qq = pd.Series(oof).rank(pct=True, method="first")
tier_hist = pd.cut(qq, [0, 0.6, 0.85, 1], labels=["Low", "Medium", "High"], include_lowest=True)
hist_tier = pd.DataFrame({"y": y, "t": tier_hist}).groupby("t").y.agg(["size", "sum", "mean"])
print("AUC grouped CV", round(auc_cv, 3), "rolling", np.round(aucs, 2), round(np.mean(aucs), 3))
print(hist_tier)
model = LogisticRegression(C=0.3, max_iter=2000).fit(F[cols], y)
cs = E[E.status.isin(["Active", "Leave of Absence"])].copy()
cs["ten0"] = (SNAP - cs["Date of Hire"]).dt.days / 365.25
cs["age0"] = (SNAP - cs.DOB).dt.days / 365.25
Fc = feats(cs)
cs["risk"] = model.predict_proba(Fc[cols])[:, 1]
cs["tier"] = pd.qcut(cs.risk.rank(method="first"), [0, 0.6, 0.85, 1], labels=["Low", "Medium", "High"]).astype(str)
cs.to_pickle("cur.pkl")
tg = cs.groupby("tier").agg(n=("risk", "size"), avg=("risk", "mean"), exp=("risk", "sum"))
print(tg, cs.risk.sum(), cs.risk.mean(), P.y.mean())
hi = cs[cs.tier == "High"]
prof = {c: (Fc.loc[hi.index, c].mean(), Fc[c].mean()) for c in cols}
prof["n_high"] = len(hi)
print({k: (np.round(v, 2) if not isinstance(v, int) else v) for k, v in prof.items()})
print(pd.crosstab(cs.tier, cs.Department, normalize="index").round(2).to_string())
print(
    cs.groupby("tier")
    .apply(
        lambda gg: pd.Series(
            {
                "single": (gg.MaritalDesc == "Single").mean(),
                "married": (gg.MaritalDesc == "Married").mean(),
                "age": gg.age0.mean(),
            }
        )
    )
    .round(2)
)
base = P.y.mean()
seg = {
    "Production staff": P.Department == "Production",
    "Non-production staff": P.Department != "Production",
    "Age 50+": P.age0 >= 50,
    "Age under 50": P.age0 < 50,
    "Divorced": P.MaritalDesc == "Divorced",
    "Married": P.MaritalDesc == "Married",
    "Single": P.MaritalDesc == "Single",
    "Tenure 3+ years": P.ten0 >= 3,
    "Tenure under 3 years": P.ten0 < 3,
    "Paid >3% below job-title median": P.pay_vs_pos < 0.97,
    "Paid at/above job-title median": P.pay_vs_pos >= 0.97,
    "Rating: Needs Improvement / PIP": P["Performance Score"].isin(["Needs Improvement", "PIP"]),
    "Rating: 90-day meets": P["Performance Score"] == "90-day meets",
    "Rating: Exceeds / Exceptional": P["Performance Score"].isin(["Exceeds", "Exceptional"]),
}
SEG = pd.DataFrame(
    [(k, P.y[m_].mean(), int(m_.sum()), int(P.y[m_].sum())) for k, m_ in seg.items()], columns=["seg", "p", "n", "ev"]
)
pickle.dump(SEG, open("attr_seg.pkl", "wb"))
print(SEG.round(4))
order = SEG.sort_values("p", ascending=False).seg.tolist()
order = [
    "Age 50+",
    "Rating: 90-day meets",
    "Divorced",
    "Rating: Needs Improvement / PIP",
    "Tenure 3+ years",
    "Paid >3% below job-title median",
    "Married",
    "Production staff",
    "Age under 50",
    "Single",
    "Non-production staff",
    "Rating: Exceeds / Exceptional",
]
d2 = SEG.set_index("seg").loc[order]
fig, ax = plt.subplots(figsize=(7.2, 3.9))
cl = [ORG if p > base * 1.3 else (TEAL if p < base * 0.85 else GRY) for p in d2.p]
ax.barh(d2.index[::-1], d2.p[::-1] * 100, color=cl[::-1])
ax.axvline(base * 100, color="k", ls="--", lw=0.9, zorder=0)
ax.text(base * 100 + 0.1, len(d2) - 0.3, f"company average {base*100:.1f}%", fontsize=7.5)
for i, (p, n, e_) in enumerate(zip(d2.p[::-1], d2.n[::-1], d2.ev[::-1])):
    ax.text(
        p * 100 + 0.12,
        i,
        f"{p*100:.1f}%  ({e_} exits / {n} person-quarters)",
        va="center",
        fontsize=7.5,
        bbox=dict(fc="white", ec="none", pad=0.8),
    )
ax.set_xlim(0, 9.5)
ax.set_xlabel("Chance of leaving within the next 3 months (historical, 2011-2016)")
ax.set_title("Chance of leaving within 3 months, by employee group")
plt.tight_layout()
plt.savefig("charts/c7_attr.png", dpi=200)
plt.close()
print("P rows", len(P), "events", P.y.sum())

# ============ 6. Assignment 2 benchmark chart ============
rows_ = [2310, 4310, 8310]
o_ = [47.993, 207.184, 462.725]
p_ = [0.346, 0.747, 1.348]
fig, ax = plt.subplots(figsize=(7.2, 2.7))
ax.plot(rows_, o_, marker="o", color=ORG, lw=2, label="Original query")
ax.plot(rows_, p_, marker="o", color=TEAL, lw=2, label="Optimised query")
for r, a, b in zip(rows_, o_, p_):
    ax.annotate(f"{a:.1f}s", (r, a), textcoords="offset points", xytext=(-4, 7), ha="right", color=ORG, fontsize=8)
    ax.annotate(f"{b:.2f}s", (r, b), textcoords="offset points", xytext=(0, -13), ha="center", color=TEAL, fontsize=8)
ax.set_xlabel("Rows in staff_particulars")
ax.set_ylabel("Execution time (s)")
ax.set_title("Execution time vs data size (lower is better)")
ax.legend(frameon=False, loc="upper left")
ax.set_ylim(-6, 700)
plt.tight_layout()
plt.savefig("charts/c8_bench.png", dpi=200)

# ============ 7. REPORT TABLES (data-quality facts, scorecards, grid, risk tiers) ============
mths = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
dq = {}
dq["duplicate Employee Number rows"] = d[d["Employee Number"].duplicated(keep=False)][
    ["Employee Name", "Employee Number"]
]
dq["Active but hired after reference date"] = int(((d.status == "Active") & (d["Date of Hire"] > SNAP)).sum())
dq["Future Start with past hire date"] = int(((d.status == "Future Start") & (d["Date of Hire"] <= SNAP)).sum())
dq["recruitment Total != monthly sum"] = rc.assign(resum=rc[mths].sum(axis=1)).query("Total != resum")[
    ["Employment Source", "Total", "resum"]
]
dq["employees whose source has no cost row"] = d[~d["Employee Source"].isin(rc["Employment Source"])][
    "Employee Source"
].value_counts()
dq["Hispanic/Latino raw values"] = pd.read_excel(X, "staff_particulars")["Hispanic/Latino"].value_counts()
dq["employees in titles without salary grid"] = int((~E.Position.isin(sg.Position)).sum())
dq["IT Manager - DB / SW Eng Manager pay"] = E[E.Position.isin(["IT Manager - DB", "Software Engineering Manager"])][
    ["Employee Name", "Position", "Pay Rate"]
]
for k, v in dq.items():
    print("\n##", k)
    print(v)
# Table: source scorecard
verdict = lambda s: "Invest" if s >= 65 else ("Keep / monitor" if s >= 45 else "Review")
tbl_sources = R["src"][["n", "oe", "perf", "cph", "score"]].assign(verdict=R["src"].score.map(verdict)).round(2)
print("\n## Source scorecard\n", tbl_sources)
# Table: manager scorecard
tbl_mgr = R["mgr"][["dept", "n", "obs", "exp", "oe", "perf", "prob", "score100"]].round(2)
print("\n## Manager scorecard\n", tbl_mgr)
# Table: pay-grid position and risk tiers
print("\n## Pay vs grid\n", pd.Series(R["grid"]))
print("\n## Risk tiers\n", tg)

/tmp/ipykernel_1143/1894091302.py:280: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mg["prob"] = M.groupby("Manager Name").apply(lambda g: ((g.lo == 1) | (g.cause == 1)).mean())
/tmp/ipykernel_1143/1894091302.py:335: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  k: smf.ols("pay ~ C(Position)+" + v, P).fit().rsquared - base.rsquared
/tmp/ipykernel_1143/1894091302.py:335: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  k: smf.ols("pay ~ C(Position)+" + v, P).fit().rsquared - base.rsquared


{'years': [2010, 2011, 2012, 2013, 2014, 2015, 2016], 'hc': [20, 90, 117, 146, 192, 200, 188], 'hires': [9, 84, 44, 44, 60, 34, 2], 'exits': [2, 14, 17, 15, 14, 26, 14], 'vex': [2, 13, 16, 14, 12, 21, 10]}
{'Female': 0.5691489361702128, 'Male': 0.4308510638297872} 36.894330610045586 2.7132101300479126 0.8457446808510638
                        n  cur  vol  cause     pay    fem  vol_rate
Department                                                         
Production            200  117   75      8  23.072  0.600     0.375
Admin Offices          10    8    2      0  31.896  0.600     0.200
Software Engineering   10    7    2      1  48.665  0.600     0.200
IT/IS                  39   29    6      4  44.138  0.462     0.154
Sales                  30   26    3      1  55.542  0.500     0.100
Executive Office        1    1    0      0  80.000  1.000     0.000
1.7727583846680357 {'Admin Offices': 3.2388774811772763, 'IT/IS': 1.0308008213552362, 'Production': 1.648186173853525, 'Sales': 2.9650

/tmp/ipykernel_1143/1894091302.py:472: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  hist_tier = pd.DataFrame({"y": y, "t": tier_hist}).groupby("t").y.agg(["size", "sum", "mean"])
/tmp/ipykernel_1143/1894091302.py:492: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


AUC grouped CV 0.637 rolling [0.64 0.56 0.49 0.71] 0.602
        size  sum      mean
t                          
Low     1686   31  0.018387
Medium   702   27  0.038462
High     422   22  0.052133
          n       avg       exp
tier                           
High     29  0.055675  1.614582
Low     113  0.021371  2.414886
Medium   46  0.034860  1.603540 5.633008514579457 0.029962811247763066 0.028469750889679714
{'production': array([0.86, 0.62]), 'tenure_3y_plus': array([0.55, 0.4 ]), 'age_50_plus': array([0.21, 0.08]), 'divorced': array([0.34, 0.07]), 'married': array([0.45, 0.37]), 'pay_below_pos_median': array([0.55, 0.29]), 'perf_weak': array([0.1 , 0.06]), 'perf_90day': array([0.38, 0.1 ]), 'n_high': 29}
Department  Admin Offices  Executive Office  IT/IS  Production  Sales  Software Engineering
tier                                                                                       
High                 0.00              0.00   0.14        0.86   0.00                  0.00
Low